In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchdiffeq import odeint
import matplotlib.pyplot as plt

Area = 3.1415926 * 0.0035 * 0.0035
eps  = 2.2204e-8

class PFR(nn.Module):
    def __init__(self):
        super(PFR,self).__init__()
        self.rho = nn.Sequential(nn.Linear(1, 5), nn.Tanh(), nn.Linear(5, 1), nn.Sigmoid()) 
        
    def forward(self, t, y):
        
        F1, F2, F3, F4, F5, F6, F7, F8, F9, F10, F11, F12 = y[...,:1], y[...,1:2], y[...,2:3], y[...,3:4], y[...,4:5], y[...,5:6], y[...,6:7], y[...,7:8], y[...,8:9], y[...,9:10], y[...,10:11], y[...,11:]
        
        R_const             = 8.3145
        pi                  = 3.1415926        
        k1F                 = 0.032/893
        k1R                 = 0.258/893
        k2                  = 0.006/245
        k3                  = 0.035/893
        k4                  = 0.0055/245
        k8                  = 0.0075/893
        n11F                = 0.65
        n12R                = 1.15
        n13R                = 0.45
        n21                 = 1.05
        n22                 = 0.45
        n31                 = 1.9
        n32                 = 1.1
        n41                 = 1.2
        n42                 = 1.2
        n81                 = 0.6
        R_r                 = 0.0035
        rho_bed             = 455
        T0                  = 623.15
        Area                = pi * R_r * R_r
        n_Cu                = 2.25
        n_Cs                = 2
        One_selectivity     = 0.56
        Two_selectivity     = 0.8
        Four_selectivity    = 0.8
        Five_One            = (1 - One_selectivity) / One_selectivity
        Six_Two             = (1 - Two_selectivity) / Two_selectivity
        Seven_Four          = (1 - Four_selectivity) / Four_selectivity
        P0                  = 170000
        F13                 = 1.1462E-5/Area

        Cu_massratio = self.rho(t.view(-1, 1)).squeeze() * 0.06 + 0.25
        Cs_massratio = 1 - Cu_massratio
        Cu_mass = (Cu_massratio * rho_bed) ** n_Cu
        Cs_mass = (Cs_massratio * rho_bed) ** n_Cs

        F_tot = F1 + F2 + F3 + F4 + F5 + F6 + F7 + F8 + F9 + F10 + F11 + F12 + F13
        C1 = F1 * P0 / T0 / R_const / F_tot 
        C2 = F2 * P0 / T0 / R_const / F_tot
        C3 = F3 * P0 / T0 / R_const / F_tot
        C5 = F5 * P0 / T0 / R_const / F_tot
        C6 = F6 * P0 / T0 / R_const / F_tot
        C9 = F9 * P0 / T0 / R_const / F_tot

        r1 = Cu_mass * (k1F * C1**n11F - k1R * C2**n12R * C3**n13R) 
        r2 = Cs_mass * k2 * C2**n21 * C5**n22
        r3 = Cu_mass * k3 * C3**n31 * C6**n32
        r4 = Cs_mass * k4 * C2**n41 * C9**n42
        r5 = Five_One * r1
        r6 = Six_Two * r2
        r7 = Seven_Four * r4
        r8 = Cu_mass * k8 * C2**n81

        dF1dz =  -r1 - r5
        dF2dz =   r1 - r2 - r4 - r6 - r7 - r8
        dF3dz =   r1 - r3 + r8      
        dF4dz =   r5
        dF5dz =  -r2 - r6
        dF6dz =   r2 - r3
        dF7dz =   r2 + r4
        dF8dz =   r6
        dF9dz =   r3 - r4 - r7
        dF10dz =  r4
        dF11dz =  r7
        dF12dz =  r8

        return torch.cat([dF1dz, dF2dz, dF3dz, dF4dz, dF5dz, dF6dz,dF7dz, dF8dz, dF9dz, dF10dz, dF11dz, dF12dz])

class NeuralODE(nn.Module):
    def __init__(self, odefunc):
        super(NeuralODE, self).__init__()
        self.odefunc = odefunc

    def forward(self, y0, t):
        sol = odeint(self.odefunc, y0, t, method="dopri8")
        return sol
    
odefunc = PFR()
z_span  = torch.linspace(0, 0.04, 10)
F0      = torch.tensor([0.1975, eps, eps, eps, 0.0987, eps, eps, eps, eps, eps, eps, eps])
model   = NeuralODE(odefunc)
model.odefunc.load_state_dict(torch.load('model_params.pth'))
soln    = model(F0, z_span)

optimizer = optim.Adam(model.parameters(), lr=0.0005)         
for epoch in range(10):                                     
    optimizer.zero_grad()
    loss = - model(F0, z_span)[-1][9]
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"{epoch}, Loss: {loss.item():.6f}")
    MMA_yield = model(F0,z_span)[-1][9]/0.0987

t_all = torch.linspace(0, 0.04, 100).view(-1, 1)
with torch.no_grad():
    Cu_massratio = model.odefunc.rho(t_all) * 0.06 + 0.25

plt.plot(t_all.numpy(), Cu_massratio.numpy())
plt.xlabel("Reactor length (x)")
plt.ylabel("ρ (Cu_massratio)")
plt.title("ρ distribution along reactor")
plt.show()

torch.save(model.state_dict(), 'model_params_final.pth')
print(MMA_yield)